# Pulse 2026: Is MAD the Right Measure of Score Reliability?

This is a focused appendix to the primary analysis. It explains why MAD is useful for describing
observed month-to-month movement but is not the criterion used to determine whether an IQB estimate
meets the 95% sampling-error requirement in `03-sample_size_reliability.ipynb`.

Notebooks 03 and 04 both use the **median absolute deviation** to describe how much an IQB score
moves. This notebook asks whether that was the right choice, by computing MAD alongside its
alternatives on the same data and showing where each one succeeds and fails.

**Short answer.** MAD is the right statistic for *describing* what usually happens to a country's
score, and the wrong statistic for *deciding* whether that score can be published. The two jobs are
different and this notebook separates them.

Two different MADs appear across the analysis, and both are examined here:

| | what it measures | where it is used |
|---|---|---|
| **predicted MAD** | spread of the sampling distribution at a given sample size | notebook 03 §7, `mad_at_current_n` |
| **observed MAD** | spread of a country's actual monthly IQB series | notebook 03 §11, notebook 04 |

---

## What MAD is

For a set of values, MAD is the median distance from the median:

$$\mathrm{MAD} = \mathrm{median}\bigl(\,|x_i - \mathrm{median}(x)|\,\bigr)$$

Half the values fall within that distance of the middle. It is the robust counterpart to a standard
deviation: the SD squares every distance, so one extreme value can dominate it, while MAD ignores
anything beyond the middle half.


## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycountry
import pyarrow.parquet as pq
from scipy import stats

from iqb.calculator.config import IQB_CONFIG

DATA_DIR = Path("./data/v1")
FIG_DIR = Path("./analysis_outputs/mad_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

PERCENTILES = [1, 5, 10, 25, 50, 75, 90, 95, 99]
QLEV = np.array(PERCENTILES) / 100.0
METRICS = ["download", "upload", "latency", "loss"]
Q_HEADLINE, TOL, CONF, TINY = 0.50, 0.05, 0.95, 1e-9

RAW_KEY = {"download": "download_throughput_mbps", "upload": "upload_throughput_mbps",
           "latency": "latency_ms", "loss": "packet_loss"}
HIGHER_BETTER = {"download": True, "upload": True, "latency": False, "loss": False}
BACKED_BY = {"download": "down", "upload": "up", "latency": "down", "loss": "down"}

INK, ACCENT, WARN, GOOD, MUTED = "#1a202c", "#2b6cb0", "#c53030", "#38a169", "#718096"
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25,
                     "grid.linestyle": "--", "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})


def save(fig, name):
    p = FIG_DIR / f"{name}.png"
    fig.savefig(p, dpi=200, bbox_inches="tight", facecolor="white")
    print(f"  saved {p}")


def country_name(cc):
    c = pycountry.countries.get(alpha_2=cc)
    return c.name if c else cc


print("comparing MAD against SD, mean absolute error, IQR and P(|error| <= tol)")

In [ ]:
def discover_periods(root: Path):
    out = []
    for pdir in sorted(root.iterdir()):
        if not pdir.is_dir():
            continue
        for edir in sorted(pdir.iterdir()):
            dl, ul = (edir / "downloads_by_country" / "data.parquet",
                      edir / "uploads_by_country" / "data.parquet")
            if dl.exists() and ul.exists():
                out.append((f"{pdir.name[:4]}-{pdir.name[4:6]}", dl, ul))
    return out


def load_all(root: Path = DATA_DIR) -> pd.DataFrame:
    frames = []
    for month, dlp, ulp in discover_periods(root):
        if dlp.stat().st_size < 1000 or ulp.stat().st_size < 1000:
            continue
        dl, ul = pq.read_table(dlp).to_pandas(), pq.read_table(ulp).to_pandas()
        m = dl.merge(ul, on="country_code", suffixes=("_down", "_up"))
        m["n_down"] = m["sample_count_down"].astype(float)
        m["n_up"] = m["sample_count_up"].astype(float)
        m["up_ratio"] = m["n_up"] / m["n_down"].clip(lower=1)
        m["month"] = month
        frames.append(m)
    return pd.concat(frames, ignore_index=True)


def derive_atoms(cfg=IQB_CONFIG):
    ucw = sum(uc["w"] for uc in cfg["use cases"].values())
    agg = {}
    for uc in cfg["use cases"].values():
        nrw = sum(nr["w"] for nr in uc["network requirements"].values())
        for nr_name, nr in uc["network requirements"].items():
            short = next(k for k, v in RAW_KEY.items() if v == nr_name)
            agg[(short, nr["threshold min"])] = (agg.get((short, nr["threshold min"]), 0.0)
                                                 + (uc["w"] / ucw) * (nr["w"] / nrw))
    out = {m: [] for m in METRICS}
    for (m, thr), w in agg.items():
        out[m].append((thr, w))
    for m in out:
        out[m].sort(key=lambda t: t[0], reverse=not HIGHER_BETTER[m])
    return out


def to_quality(metric, v):
    v = np.asarray(v, dtype=float)
    if metric in ("download", "upload"):
        return np.log(np.maximum(v, TINY))
    if metric == "latency":
        return -np.log(np.maximum(v, TINY))
    return -v


def pass_prob(y_knots, t):
    y = np.asarray(y_knots, float)
    uy = np.unique(y)
    q_lo = np.array([QLEV[y == v].min() for v in uy])
    q_hi = np.array([QLEV[y == v].max() for v in uy])
    if t <= uy[0]:
        return 0.995
    if t >= uy[-1]:
        return 0.005
    i = np.searchsorted(uy, t) - 1
    F = q_hi[i] + (q_lo[i + 1] - q_hi[i]) * (t - uy[i]) / (uy[i + 1] - uy[i])
    return float(np.clip(1.0 - F, 0.005, 0.995))


def pass_prob_matrix(frame) -> np.ndarray:
    cols = []
    for m in METRICS:
        Y = to_quality(m, frame[[f"{m}_p{p}" for p in PERCENTILES]].to_numpy(float))
        for thr, _ in ATOMS[m]:
            t = float(to_quality(m, thr))
            cols.append(np.array([pass_prob(Y[i], t) for i in range(len(Y))]))
    return np.column_stack(cols)


df = load_all()
ATOMS = derive_atoms()
ATOM_LABELS = [f"{m} {'>' if HIGHER_BETTER[m] else '<'} {thr}"
               for m in METRICS for thr, _ in ATOMS[m]]
ATOM_W = np.array([w for m in METRICS for _, w in ATOMS[m]])
BANDS = {m: np.concatenate([[0.0], np.cumsum([w for _, w in ATOMS[m]])]) for m in METRICS}
KS = [len(ATOMS[m]) for m in METRICS]
LATTICE = sum(g.ravel() for g in np.meshgrid(*[BANDS[m] for m in METRICS], indexing="ij"))
LAT_ORDER = np.argsort(LATTICE)

cc_arr = df.country_code.to_numpy()
n_arr = df.n_down.to_numpy(float)
nup_arr = df.n_up.to_numpy(float)
P_all = pass_prob_matrix(df)

print(f"{len(df):,} country-months, {df.country_code.nunique()} countries")
print(f"IQB estimates live on a {len(LATTICE)}-point lattice - the estimator is DISCRETE, which "
      f"is\nthe fact every argument in this notebook turns on.")

In [ ]:
def sampling_dist(C, q, n, ratio=1.0):
    rows = len(C)
    n_dn = np.full(rows, max(int(n), 1))
    n_up = np.maximum(np.round(n_dn * np.broadcast_to(np.asarray(ratio, float), (rows,))), 1)
    parts, off = [], 0
    for m, K in zip(METRICS, KS):
        n_m = (n_up if BACKED_BY[m] == "up" else n_dn).astype(np.int64)
        r = np.clip(np.ceil(q * n_m), 1, n_m).astype(np.int64)
        s = stats.beta.sf(C[:, off:off + K], r[:, None], (n_m - r + 1)[:, None])
        off += K
        p = np.empty((rows, K + 1))
        p[:, 0] = 1.0 - s[:, 0]
        for j in range(1, K):
            p[:, j] = s[:, j - 1] - s[:, j]
        p[:, K] = s[:, K - 1]
        parts.append(np.clip(p, 0.0, None))
    J = parts[0]
    for p in parts[1:]:
        J = (J[:, :, None] * p[:, None, :]).reshape(rows, -1)
    return J


def true_scores(P, q):
    t = np.zeros(len(P))
    for k in range(P.shape[1]):
        t += ATOM_W[k] * (q > 1.0 - P[:, k])
    return t


def all_stats(C, t, q, n, ratio=1.0, tol=TOL):
    """Every candidate statistic, computed exactly from the sampling distribution."""
    J = sampling_dist(C, q, n, ratio)
    # Guard: within a metric the thresholds are nested, so the cut points must be non-decreasing.
    # If they are not, sampling_dist returns negative interval probabilities that get clipped away
    # and the result is no longer a distribution - every statistic below would be meaningless.
    mass = J.sum(axis=1)
    if not np.allclose(mass, 1.0, atol=1e-6):
        raise ValueError(f"not a distribution: probability mass ranges "
                         f"{mass.min():.4f}..{mass.max():.4f}. Cut points are probably not nested.")
    mean = (J * LATTICE).sum(axis=1)
    sd = np.sqrt((J * (LATTICE[None, :] - mean[:, None]) ** 2).sum(axis=1))
    mae = (J * np.abs(LATTICE[None, :] - t[:, None])).sum(axis=1)
    pok = (J * (np.abs(LATTICE[None, :] - t[:, None]) <= tol + 1e-12)).sum(axis=1)

    L, Js = LATTICE[LAT_ORDER], J[:, LAT_ORDER]
    cdf = np.cumsum(Js, axis=1) / Js.sum(axis=1, keepdims=True)
    med = L[np.argmax(cdf >= 0.5, axis=1)]
    q25 = L[np.argmax(cdf >= 0.25, axis=1)]
    q75 = L[np.argmax(cdf >= 0.75, axis=1)]
    d = np.abs(L[None, :] - med[:, None])
    order = np.argsort(d, axis=1)
    dc = np.take_along_axis(Js, order, axis=1).cumsum(axis=1)
    dc /= dc[:, -1:]
    mad = np.take_along_axis(d, order, axis=1)[np.arange(len(C)), np.argmax(dc >= 0.5, axis=1)]
    return dict(mad=mad, sd=sd, mae=mae, iqr=q75 - q25, p_within=pok,
                bias=mean - t, median=med)


# Pool the 24 months to get each country's pass rates, exactly as notebook 03 does.
codes, P_rows, n_med, ratio_rows = [], [], [], []
for c in pd.unique(cc_arr):
    m = cc_arr == c
    w = n_arr[m]
    w = w / w.sum() if w.sum() > 0 else np.full(m.sum(), 1 / m.sum())
    codes.append(c)
    P_rows.append((P_all[m] * w[:, None]).sum(axis=0))
    n_med.append(np.median(n_arr[m]))
    ratio_rows.append(nup_arr[m].sum() / max(n_arr[m].sum(), 1.0))

P_c = np.array(P_rows)
C_c = 1 - P_c
n_med = np.array(n_med)
ratio_c = np.array(ratio_rows)
t_c = true_scores(P_c, Q_HEADLINE)
# Nesting check: passing a stricter threshold implies passing a looser one, so within a metric
# the pass rates must be non-increasing. sampling_dist depends on it.
off, viol = 0, 0
for m, K in zip(METRICS, KS):
    blk = P_c[:, off:off + K]
    off += K
    viol += int((np.diff(blk, axis=1) > 1e-12).any(axis=1).sum())
print(f"pooled to {len(codes)} countries; every statistic below is exact, not simulated.")
print(f"nesting check - countries whose reconstructed pass rates are not nested: {viol} / "
      f"{len(codes)}")

## 2. What Makes This Estimator Unusual

The choice of statistic only matters because the thing being summarised has an odd shape. An IQB
estimate is not a continuous quantity scattered around a centre. It is one of 144 possible values,
and at any reasonable sample size **most of its probability sits on a single point** — the correct
one — with the rest in a few isolated spikes, each one a threshold test coming out the wrong way.

The distance to the nearest spike is the weight of whichever test flipped: as little as 0.014, as
much as 0.202.

The three panels below are the actual sampling distribution for three countries. This shape is why
a standard deviation misleads, and also why MAD has the specific failure described in §4.

In [ ]:
picks = []
for lab, target in [("wide margin", 0.90), ("typical margin", 0.50), ("narrow margin", 0.03)]:
    marg = np.where(ATOM_W > TOL, np.abs(P_c - 0.5), np.inf).min(axis=1)
    ok = np.where(n_med >= 1000)[0]
    picks.append((lab, ok[np.argsort(marg[ok])[int(target * (len(ok) - 1))]]))

N_SHOW = 1000
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9), sharey=True)
for ax, (lab, i) in zip(axes, picks):
    J = sampling_dist(C_c[[i]], Q_HEADLINE, N_SHOW, ratio_c[i])[0]
    s = all_stats(C_c[[i]], t_c[[i]], Q_HEADLINE, N_SHOW, ratio_c[i])
    keep = J > 1e-4
    ax.bar(LATTICE[keep], J[keep], width=0.012, color=ACCENT, zorder=3)
    ax.axvline(t_c[i], color=WARN, lw=1.6, ls="--", zorder=4)
    ax.text(t_c[i], 1.02, " true score", color=WARN, fontsize=8.5,
            transform=ax.get_xaxis_transform(), va="bottom")
    ax.set_xlim(-0.03, 1.03)
    ax.set_xlabel("reported IQB score")
    ax.set_title(f"{codes[i]} — {lab}\nMAD {s['mad'][0]:.3f}   SD {s['sd'][0]:.3f}   "
                 f"P(within {TOL}) {s['p_within'][0]:.2f}", fontsize=9.5)
axes[0].set_ylabel(f"probability at n = {N_SHOW:,}")
fig.suptitle("The sampling distribution of an IQB score: one tall spike, a few small ones",
             fontsize=11.5)
fig.tight_layout()
save(fig, "fig1_what_the_distribution_looks_like")
plt.show()

print(f"{'country':>9s}{'P(exactly right)':>19s}{'MAD':>8s}{'SD':>8s}"
      f"{'P(within tol)':>15s}")
for lab, i in picks:
    J = sampling_dist(C_c[[i]], Q_HEADLINE, N_SHOW, ratio_c[i])[0]
    s = all_stats(C_c[[i]], t_c[[i]], Q_HEADLINE, N_SHOW, ratio_c[i])
    exact = float(J[np.isclose(LATTICE, t_c[i])].sum())
    print(f"{codes[i]:>9s}{exact:>19.3f}{s['mad'][0]:>8.3f}{s['sd'][0]:>8.3f}"
          f"{s['p_within'][0]:>15.3f}")

## 3. Where MAD Is the Right Choice

### 3a. A standard deviation reports the rare jumps, not the typical state

An SD squares every distance, so the isolated spikes above — which carry a few percent of the
probability but sit 0.1 to 0.2 away — dominate it. The result describes something that almost never
happens. MAD ignores anything past the middle half, so it describes what you actually get.

The table below makes the divergence concrete: for many countries the SD is large while the score
is, in fact, exactly right most of the time.

In [ ]:
S = {}
for n in (100, 1_000, 10_000):
    S[n] = all_stats(C_c, t_c, Q_HEADLINE, n, ratio_c)

s = S[1_000]
J = sampling_dist(C_c, Q_HEADLINE, 1_000, ratio_c)
exact = np.array([J[i][np.isclose(LATTICE, t_c[i])].sum() for i in range(len(codes))])

gap = (s["sd"] > 0.04) & (s["mad"] == 0)
print(f"At n = 1,000: {gap.sum()} of {len(codes)} countries have an SD above 0.04 while their "
      f"score is\nexactly right more than half the time. For those the SD is describing an "
      f"outcome that is rare.\n")
idx = np.argsort(-s["sd"] * gap)[:6]
print(f"{'country':>9s}{'true':>8s}{'SD':>8s}{'MAD':>7s}{'P(exactly right)':>19s}")
for i in idx:
    print(f"{codes[i]:>9s}{t_c[i]:>8.3f}{s['sd'][i]:>8.3f}{s['mad'][i]:>7.3f}{exact[i]:>19.3f}")

print(f"\nAcross all countries at n = 1,000: median SD {np.median(s['sd']):.4f}, "
      f"median MAD {np.median(s['mad']):.4f}.")
print(f"The score is exactly correct for a median of {np.median(exact):.0%} of the probability - "
      f"which is\nwhat MAD = 0 is reporting, and what the SD hides.")

### 3b. MAD = 0 has an exact, checkable meaning

Because the estimator is discrete and its most likely value is usually the true one, `MAD = 0` is
not a rounding artefact. It states something precise and falsifiable:

> **More than half the time, the published score is exactly the true score — not close to it,
> equal to it.**

No other candidate statistic has a reading that clean. An SD of 0.03 on a discrete estimator does
not correspond to any outcome that can occur; the mean absolute error of 0.008 is an average over
outcomes that are mostly 0 and occasionally 0.2. For explaining results to a non-technical reader,
this is a genuine advantage and the main reason MAD was used in notebooks 03 and 04.

### 3c. It is robust to exactly the thing that would break an SD here

One test — `loss < 0.01`, weight 0.202 — accounts for a fifth of the whole scale. If that test is even
slightly uncertain, it puts 0.202 of probability mass a fifth of the scale away, and an SD inherits
it in full. MAD has a 50% breakdown point: it is unmoved until the unreliable outcome becomes the
*majority* outcome.

In [ ]:
heavy_k = int(np.argmax(ATOM_W))
print(f"heaviest test: {ATOM_LABELS[heavy_k]}  (weight {ATOM_W[heavy_k]:.4f})\n")

# How much of the SD is contributed by the single heaviest test being uncertain?
p_flip = np.abs(sampling_dist(C_c, Q_HEADLINE, 1_000, ratio_c) @ np.ones(len(LATTICE)))  # = 1
u = np.abs(P_c[:, heavy_k] - 0.5)                      # how close that test is to its cut-off
near = u < 0.02
print(f"{near.sum()} countries sit within 0.02 of the cut-off on that one test.")
print(f"   their median SD      {np.median(S[1000]['sd'][near]):.4f}")
print(f"   their median MAD     {np.median(S[1000]['mad'][near]):.4f}")
print(f"   everyone else, SD    {np.median(S[1000]['sd'][~near]):.4f}")
print(f"   everyone else, MAD   {np.median(S[1000]['mad'][~near]):.4f}")
print("\nThe SD is dominated by that single uncertain test; MAD stays at zero until the")
print("wrong answer becomes the more likely one. Which of those you want depends on the question.")

## 4. Where MAD Fails

Four failures, in order of how much they matter. The first one is serious enough that MAD should
never be used on its own to decide whether a score is publishable.

### 4a. It saturates at zero and then stops measuring anything

MAD's floor is not a limit it approaches — it is a value it reaches and sits at. Once the correct
score is the most likely single outcome by any margin at all, MAD is exactly 0, and **it stays 0
whether the score is right 51% of the time or 99.9% of the time.**

Those two situations are not remotely equivalent for anyone deciding what to publish, and MAD
cannot tell them apart. The figure below shows the damage: among countries with `MAD = 0`, the
probability of being within tolerance ranges across most of the unit interval.

In [ ]:
s = S[1_000]
zero = s["mad"] == 0
print(f"At n = 1,000, {zero.sum()} of {len(codes)} countries have MAD = 0.")
print(f"Among those, P(within +/-{TOL}) ranges from {s['p_within'][zero].min():.3f} "
      f"to {s['p_within'][zero].max():.3f}.")
print(f"   {(s['p_within'][zero] < CONF).sum()} of them are BELOW the {CONF:.0%} bar - "
      f"MAD = 0 and still not publishable.")
print(f"   {(s['p_within'][zero] < 0.75).sum()} of them are below 75%.\n")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
ax = axes[0]
ax.scatter(s["p_within"][~zero], s["mad"][~zero], s=20, alpha=0.6, color=ACCENT,
           label="MAD > 0")
ax.scatter(s["p_within"][zero], np.full(zero.sum(), 0), s=26, alpha=0.55, color=WARN,
           marker="D", label=f"MAD = 0 ({zero.sum()} countries)")
ax.axvline(CONF, color=INK, ls="--", lw=1.3)
ax.text(CONF - 0.008, ax.get_ylim()[1] * 0.45, f"the {CONF:.0%} bar  ", ha="right",
        fontsize=8.5, color=INK)
ax.set_xlabel(f"P(|error| <= {TOL}) — the quantity the decision actually needs")
ax.set_ylabel("MAD")
ax.set_title("Every red point has MAD = 0.\nThey span most of the range that matters.",
             fontsize=10.5)
ax.legend(fontsize=8.5, loc="upper left")

ax = axes[1]
i = picks[2][1]
grid = np.unique(np.round(np.logspace(1, 5.2, 46)).astype(int))
curves = {k: [] for k in ("mad", "sd", "mae", "p_within")}
for n in grid:
    st = all_stats(C_c[[i]], t_c[[i]], Q_HEADLINE, int(n), ratio_c[i])
    for k in curves:
        curves[k].append(st[k][0])
for k, col, lab in [("mad", WARN, "MAD"), ("sd", ACCENT, "SD"), ("mae", MUTED, "mean abs error")]:
    ax.plot(grid, curves[k], lw=1.9, color=col, label=lab)
ax2 = ax.twinx()
ax2.plot(grid, curves["p_within"], lw=2.2, color=GOOD, ls="--", label=f"P(within {TOL})")
ax2.set_ylabel(f"P(|error| <= {TOL})", color=GOOD)
ax2.set_ylim(0, 1.02)
ax2.grid(False)
ax.set_xscale("log")
ax.set_xlabel("measurements")
ax.set_ylabel("spread of the estimate")
ax.set_title(f"{codes[i]}: MAD reaches zero long before\nthe score is actually reliable",
             fontsize=10.5)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="center right")
fig.tight_layout()
save(fig, "fig2_saturation")
plt.show()

first0 = grid[np.argmax(np.array(curves["mad"]) == 0)]
firstok = grid[np.argmax(np.array(curves["p_within"]) >= CONF)] if max(curves["p_within"]) >= CONF else None
print(f"For {codes[i]}: MAD first hits 0 at n = {first0:,}; "
      f"the {CONF:.0%} bar is met at n = {firstok:,}." if firstok else
      f"For {codes[i]}: MAD first hits 0 at n = {first0:,}; the bar is never met on this grid.")

### 4b. It is measured around the wrong centre, so it cannot see bias

MAD measures spread around **the estimator's own median**, not around the true score. That makes it
completely blind to bias — and notebook 03 §8b shows the bias here is real, systematic, and points
the same direction for every under-measured country.

A country whose score is reliably wrong by 0.13 has a MAD of 0. MAD reports it as maximally stable,
which is true and also exactly the wrong impression.

The same criticism applies to the **observed** MAD used in notebook 04: it is computed around each
country's own median month, so a country consistently reported below its true score looks perfectly
steady.

In [ ]:
bias_now = np.empty(len(codes))
mad_now = np.empty(len(codes))
pok_now = np.empty(len(codes))
for n in np.unique(np.maximum(n_med, 1).astype(int)):
    idx = np.where(np.maximum(n_med, 1).astype(int) == n)[0]
    st = all_stats(C_c[idx], t_c[idx], Q_HEADLINE, int(n), ratio_c[idx])
    bias_now[idx], mad_now[idx], pok_now[idx] = st["bias"], st["mad"], st["p_within"]

blind = (mad_now == 0) & (np.abs(bias_now) > TOL)
print(f"{blind.sum()} countries have MAD = 0 at their current sample size AND a bias larger than "
      f"the\n+/-{TOL} tolerance. MAD calls each of them perfectly stable.\n")
print(f"{'country':>26s}{'n/month':>10s}{'true':>8s}{'reported':>10s}{'bias':>9s}{'MAD':>7s}")
for i in np.argsort(bias_now)[:8]:
    if not blind[i]:
        continue
    print(f"{country_name(codes[i])[:25]:>26s}{n_med[i]:>10,.0f}{t_c[i]:>8.3f}"
          f"{t_c[i] + bias_now[i]:>10.3f}{bias_now[i]:>+9.3f}{mad_now[i]:>7.3f}")

fig, ax = plt.subplots(figsize=(8.2, 4.6))
ax.scatter(np.abs(bias_now[~blind]), mad_now[~blind], s=20, alpha=0.55, color=ACCENT,
           label="rest of the panel")
ax.scatter(np.abs(bias_now[blind]), mad_now[blind], s=44, alpha=0.9, color=WARN,
           label=f"MAD = 0 but biased past the tolerance ({blind.sum()})")
ax.axvline(TOL, color=INK, ls="--", lw=1.3)
ax.text(TOL * 1.08, ax.get_ylim()[1] * 0.92, f"tolerance {TOL}", fontsize=8.5, color=INK)
ax.set_xlabel("size of the bias in the published score")
ax.set_ylabel("MAD at the country's current sample size")
ax.set_title("MAD is zero for the countries whose scores are most wrong", fontsize=11)
ax.legend(fontsize=8.5)
save(fig, "fig3_blind_to_bias")
plt.show()

### 4c. The usual rescaling to a standard deviation does not apply

MAD is often multiplied by **1.4826** to put it on the same scale as a standard deviation. That
constant is derived from the normal distribution and is meaningless here — the estimator is
discrete, one-sided and mostly concentrated on a single point.

Notebook 03 §11 applies that scaling to the observed monthly series so the observed and predicted
spreads can be compared on one axis. The cell below measures what the true ratio actually is.

In [ ]:
K_NORMAL = 1.4826

# --- predicted: SD / MAD of the exact sampling distribution
pred = S[1_000]
ok = pred["mad"] > 0
ratio_pred = pred["sd"][ok] / pred["mad"][ok]

# --- observed: SD / MAD of each country's 24-month series
t_month = true_scores(P_all, Q_HEADLINE)
h = pd.DataFrame({"cc": cc_arr, "iqb": t_month}).groupby("cc")
obs = pd.DataFrame({"sd": h.iqb.std(),
                    "mad": h.iqb.apply(lambda s: float(np.median(np.abs(s - np.median(s))))),
                    "months": h.size()}).reset_index()
obs = obs[(obs.months >= 12) & (obs["mad"] > 0)]
ratio_obs = (obs.sd / obs["mad"]).to_numpy()

print(f"the constant assumed by the rescaling:   {K_NORMAL:.4f}\n")
print(f"{'':22s}{'countries':>11s}{'median':>9s}{'P10':>8s}{'P90':>8s}")
print(f"{'predicted (exact)':22s}{ok.sum():>11d}{np.median(ratio_pred):>9.2f}"
      f"{np.quantile(ratio_pred, .1):>8.2f}{np.quantile(ratio_pred, .9):>8.2f}")
print(f"{'observed (24 months)':22s}{len(ratio_obs):>11d}{np.median(ratio_obs):>9.2f}"
      f"{np.quantile(ratio_obs, .1):>8.2f}{np.quantile(ratio_obs, .9):>8.2f}")

fig, ax = plt.subplots(figsize=(8.2, 4.3))
bins = np.logspace(np.log10(0.3), np.log10(30), 40)
ax.hist(ratio_obs, bins=bins, color=ACCENT, alpha=0.75, label="observed monthly series")
ax.hist(ratio_pred, bins=bins, color=WARN, alpha=0.6, label="predicted sampling distribution")
ax.axvline(K_NORMAL, color=INK, lw=2)
ax.text(K_NORMAL * 1.06, ax.get_ylim()[1] * 0.88,
        f" {K_NORMAL} — what the\n rescaling assumes", fontsize=9, color=INK)
ax.set_xscale("log")
ax.set_xlabel("actual SD ÷ MAD")
ax.set_ylabel("countries")
ax.set_title("The 1.4826 rescaling assumes a ratio this data does not have", fontsize=11)
ax.legend(fontsize=8.5)
save(fig, "fig4_rescaling_invalid")
plt.show()

print(f"\nNote how few countries even have a defined ratio: MAD is zero for the rest, and the")
print("rescaling is undefined there - it turns a zero into a zero and says nothing.")
print(f"\nWhere it IS defined the ratio is not {K_NORMAL} and not any single number: it spans a")
print("factor of about five across countries, because it depends on which test is uncertain and")
print("how heavy that test is. A rescaled MAD is therefore not comparable to an SD, and notebook")
print("03 should report the raw MAD next to the SD rather than a rescaled one.")

### 4d. It does not respond to the thing that matters

MAD on a discrete estimator can only take values that are distances between lattice points, so it
is coarse by construction. But the sweep below shows something worse than coarseness.

A country is held fixed while the pass rate on the heaviest test is moved through its cut-off, so
the score goes from certain, to a 50/50 result, and back. `P(within tolerance)` traces that exactly,
falling from 1.00 to 0.51. The SD and the mean absolute error both rise and fall with it.

**MAD stays at exactly zero for the whole sweep.** Not approximately zero, and not stepping between
two values — one single value across all 240 points, while the score underneath goes from reliable
to a coin toss and back. A statistic that cannot move when reliability halves cannot be used to
detect, rank, or threshold on reliability.


In [ ]:
i = picks[1][1]
sweep = np.linspace(0.40, 0.60, 240)
# Both loss thresholds have to move together: passing loss < 0.005 implies passing loss < 0.01,
# so their cut points cannot cross. Sliding only the heavy one would produce something that is
# not a probability distribution - the guard in all_stats now refuses it.
gap = C_c[i, heavy_k + 1] - C_c[i, heavy_k]
Cs = np.repeat(C_c[[i]], len(sweep), axis=0)
Cs[:, heavy_k] = 1 - sweep
Cs[:, heavy_k + 1] = np.clip(Cs[:, heavy_k] + gap, 0.005, 0.995)
ts = true_scores(1 - Cs, Q_HEADLINE)
st = all_stats(Cs, ts, Q_HEADLINE, 5_000, ratio_c[i])

fig, ax = plt.subplots(figsize=(8.6, 4.5))
ax.plot(sweep, st["mad"], lw=2.4, color=WARN, label="MAD")
ax.plot(sweep, st["sd"], lw=1.8, color=ACCENT, label="SD")
ax.plot(sweep, st["mae"], lw=1.8, color=MUTED, label="mean absolute error")
ax2 = ax.twinx()
ax2.plot(sweep, st["p_within"], lw=2.2, color=GOOD, ls="--", label=f"P(within {TOL})")
ax2.set_ylabel(f"P(|error| <= {TOL})", color=GOOD)
ax2.set_ylim(-0.02, 1.02)
ax2.grid(False)
ax.axvline(0.5, color=INK, lw=1.2, ls=":")
ax.text(0.5, ax.get_ylim()[1] * 0.96, " the cut-off", fontsize=8.5, color=INK, va="top")
ax.set_xlabel(f"pass rate on the heaviest test ({ATOM_LABELS[heavy_k]}, weight "
              f"{ATOM_W[heavy_k]:.3f}); {ATOM_LABELS[heavy_k + 1]} moves with it")
ax.set_ylabel("spread of the estimate")
ax.set_title(f"{codes[i]} at n = 5,000: MAD never moves; everything else tracks the score",
             fontsize=11)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8.5, loc="center left")
save(fig, "fig5_mad_does_not_respond")
plt.show()

print(f"across the {len(sweep)} points of that sweep:")
for k, lab in [("p_within", f"P(within {TOL})"), ("sd", "SD"), ("mae", "mean abs error"),
               ("mad", "MAD")]:
    v = st[k]
    print(f"   {lab:18s} range {v.min():.4f} .. {v.max():.4f}   "
          f"{len(np.unique(np.round(v, 6))):>3d} distinct values")
w = int(np.argmin(st["p_within"]))
print(f"\nAt the worst point the score is right only {st['p_within'][w]:.0%} of the time. "
      f"MAD there: {st['mad'][w]:.4f}.\n")

vals = np.unique(np.round(S[1_000]["mad"], 6))
print(f"distinct MAD values taken across all {len(codes)} countries at n = 1,000: {len(vals)}")
print(f"   {', '.join(f'{v:.4f}' for v in vals[:10])}{' ...' if len(vals) > 10 else ''}")
print(f"\nBy comparison, P(within {TOL}) takes {len(np.unique(np.round(S[1_000]['p_within'], 4)))}"
      f" distinct values over the same countries.")
print("MAD cannot order countries that share a value, which is most of them.")

### 4e. The decision is about the tail, and MAD is a statement about the middle

The question the analysis exists to answer is *"can this score be published?"*, which is a question
about how often the score is badly wrong — the tail. MAD describes the middle half of the
distribution and is silent about the rest by construction. That is not a defect in MAD; it is a
mismatch between the statistic and the question.

This is why notebook 03 sets its requirement with `P(|error| ≤ 0.05) ≥ 0.95` and uses MAD only to
describe. Getting that division right is the whole point of this notebook.

## 5. The Candidates, Side by Side

In [ ]:
rows = []
for n in (100, 1_000, 10_000):
    st = S[n]
    rows.append({"n": f"{n:,}", "MAD": np.median(st["mad"]), "IQR": np.median(st["iqr"]),
                 "mean abs err": np.median(st["mae"]), "SD": np.median(st["sd"]),
                 "P(within tol)": np.median(st["p_within"]),
                 "|bias|": np.median(np.abs(st["bias"]))})
print("median across the 245 countries")
print(pd.DataFrame(rows).set_index("n").to_string(float_format=lambda v: f"{v:.4f}"))

print(f"\n{'statistic':18s}{'saturates?':>12s}{'sees bias?':>12s}{'smooth?':>10s}"
      f"{'answers the decision?':>23s}")
table = [("MAD", "yes, at 0", "no", "no", "no"),
         ("IQR", "yes, at 0", "no", "no", "no"),
         ("SD", "no", "no", "yes", "no"),
         ("mean abs error", "no", "yes", "yes", "no"),
         ("P(within tol)", "at 1", "yes", "yes", "yes")]
for r in table:
    print(f"{r[0]:18s}{r[1]:>12s}{r[2]:>12s}{r[3]:>10s}{r[4]:>23s}")

print("\nMean absolute error is the quiet winner among the spread measures: it is taken around the")
print("TRUE score rather than the estimator's median, so unlike MAD, SD and IQR it does account")
print("for bias. It is harder to explain, which is why MAD survived in the reporting.")

corr = pd.DataFrame({k: S[1_000][k] for k in ("mad", "iqr", "mae", "sd", "p_within")}).corr(
    method="spearman")
print(f"\nrank correlation with P(within {TOL}) at n = 1,000:")
for k in ("mad", "iqr", "sd", "mae"):
    print(f"   {k:16s}{corr.loc[k, 'p_within']:+.3f}")

## 6. Conclusion

**MAD was the right choice for what it was used for, and would be the wrong choice for anything
else.** The division that matters:

| job | statistic | why |
|---|---|---|
| Describing what usually happens | **MAD** | The estimator is discrete and usually exactly right. `MAD = 0` says "more than half the time the published score is exactly correct", which is true, checkable, and explainable to a non-specialist. |
| Deciding whether to publish | **P(\|error\| ≤ τ)** | The decision is about the tail. MAD saturates at 0 while that probability is still anywhere between 0.5 and 1.0. |
| Reporting how wrong a score is | **bias, separately** | MAD, IQR and SD are all measured around the estimator's own centre and are blind to bias by construction. |

### Three concrete changes

1. **Never publish MAD on its own.** A country with `MAD = 0` may still be outside tolerance
   nearly half the time, and may be biased by more than the tolerance itself. Every place MAD
   appears, `p_ok_at_current_n` and `bias_at_current_n` should appear beside it — as they do in
   `analysis_outputs/03_sample_size_requirements.csv`.

2. **Drop the 1.4826 rescaling in notebook 03 §11.** The constant assumes a normal distribution.
   The actual ratio here varies by an order of magnitude across countries, so a rescaled MAD is not
   comparable to an SD and the rescaling only obscures the raw number.

3. **Do not use MAD to rank or threshold countries.** It takes eight distinct values across the
   whole panel, and in the sweep in §4d it does not move at all while the score goes from certain
   to a 50/50 result. Rank on
   `P(|error| ≤ τ)`, which is smooth and is the quantity the decision actually depends on. If a
   spread measure must be used for ranking, note that the ordinary **standard deviation ranks
   countries far better than MAD does** — rank correlation with `P(|error| ≤ τ)` is −0.95 for the
   SD against −0.28 for MAD. MAD's robustness, which is its virtue when describing, is precisely
   what destroys its ranking power.

### What would change the verdict

If IQB were rescored so the thresholds ramped rather than stepped, the estimator would become
continuous, and MAD would lose both its main advantage (robustness to a discrete jump structure)
and its main defect (saturation at zero). At that point an ordinary standard deviation would be
the natural choice. The case for MAD here rests entirely on IQB being a sum of hard indicators.


In [ ]:
out = pd.DataFrame({
    "cc": codes, "country": [country_name(c) for c in codes],
    "n_down_month_median": n_med, "iqb_p50": t_c,
    "mad_at_current_n": mad_now, "bias_at_current_n": bias_now,
    "p_ok_at_current_n": pok_now,
    "mad_is_zero_but_unreliable": (mad_now == 0) & (pok_now < CONF),
    "mad_is_zero_but_biased": (mad_now == 0) & (np.abs(bias_now) > TOL),
})
p = Path("./analysis_outputs/06_mad_diagnostics.csv")
out.sort_values("p_ok_at_current_n").to_csv(p, index=False)
print(f"wrote {p}\n")
print(f"countries where MAD = 0 but the score is not publishable: "
      f"{int(out.mad_is_zero_but_unreliable.sum())}")
print(f"countries where MAD = 0 but the score is biased past the tolerance: "
      f"{int(out.mad_is_zero_but_biased.sum())}")
print(f"\nfigures written to {FIG_DIR}/:")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"   {f.name}")